# Fraude Pagamentos RTR - Analysis Notebook

## Objetivo
Classificar transacoes suspeitas para reduzir perda financeira e manter experiencia do cliente.

## Perguntas de negocio
- Qual trade-off entre recall e precision?
- Quais sinais explicam maior parte do risco?
- Threshold atual esta alinhado ao custo da operacao?


### O que este código faz
Carrega bibliotecas, lê os arquivos principais do projeto (`data` e `metrics`) e mostra uma amostra das transações para validação inicial.


In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

project_root = Path('..')
data_path = project_root / 'data' / 'transactions_synthetic.csv'
metrics_path = project_root / 'models' / 'metrics.json'

pd.set_option('display.max_columns', 200)
df = pd.read_csv(data_path)
metrics = json.loads(metrics_path.read_text(encoding='utf-8')) if metrics_path.exists() else {}

print('shape:', df.shape)
print('metrics keys:', list(metrics.keys()))
df.head(5)


### O que este código faz
Faz checagem rápida de qualidade dos dados (faltantes e distribuição do alvo de fraude).


In [ ]:
missing = df.isna().sum().sort_values(ascending=False)
missing = missing[missing > 0]
print('missing columns:', len(missing))
if len(missing) > 0:
    display(missing)

if 'is_fraud' in df.columns:
    print('target distribution:')
    print(df['is_fraud'].value_counts(normalize=True).round(4))


### O que este código faz
Compara as variáveis de risco entre transações legítimas e fraude para identificar sinais mais relevantes.


In [ ]:
risk_cols = [c for c in ['amount', 'velocity_1h', 'velocity_24h', 'device_risk', 'merchant_risk'] if c in df.columns]
if risk_cols:
    fig, axes = plt.subplots(1, len(risk_cols), figsize=(4 * len(risk_cols), 3.5))
    if len(risk_cols) == 1:
        axes = [axes]
    for ax, col in zip(axes, risk_cols):
        ax.boxplot([df.loc[df['is_fraud']==0, col], df.loc[df['is_fraud']==1, col]], labels=['legit', 'fraud'])
        ax.set_title(col)
        ax.grid(alpha=0.2)
    plt.tight_layout()


### O que este código faz
Mostra as métricas finais do pipeline para avaliar a performance do modelo/regras.


In [ ]:
pd.DataFrame([metrics]).T.rename(columns={0: 'value'}).head(30)


## Status atual
- Este projeto ja foi executado de ponta a ponta.
- Artefatos (dados, modelos, metricas e relatorios) estao gerados.
- Repositorio publicado e sincronizado no GitHub.
